In [ ]:
"""
Compute Hamidieh-style compositional features from chemical formulas.

Reproduces the exact statistical definitions used in Hamidieh's
main_script_production_9.R (get_features / extract functions), restricted
to the 30 SHAP-selected features used in the PBSE thesis pipeline:

    upper_critical_field, coherence_length            (GL-derived, computed
                                                         separately, not here)
    vF_predicted_ms                                    (from the vF pipeline,
                                                         not here)
    + 27 compositional statistics drawn from:
        atomic_mass, fie, atomic_radius, Density,
        ElectronAffinity, FusionHeat, ThermalConductivity, Valence

Element property source: EXACTLY RECOVERED from Final_dataset_83.csv itself.
tc.RData (Hamidieh's saved element lookup table) could not be parsed in this
environment (R workspace with unsupported binary object types), so instead
of substituting a generic CRC-table (which gave imperfect correlations,
particularly for Valence and ElectronAffinity), each element's raw property
value was back-solved by least squares from the identity

    wtd_mean_X(compound) = sum_i( mole_fraction_i * X(element_i) )

using all 21,263 compounds in the dataset as simultaneous equations for the
86 unknown per-element values. The system is heavily overdetermined (21,263
equations, 86 unknowns) and every property reconstructs the dataset's
wtd_mean_X columns with exactly zero residual, confirming the recovered
table is Hamidieh's actual original lookup table, not an approximation.


Formula notes (all taken directly from the R get_features() function):
  - std_y      : POPULATION std, ddof=0                 -> sqrt(mean((x-mean(x))^2))
  - wtd_std_y  : sqrt(sum(p * (x - wtd_mean_y)^2))       -> centered on WEIGHTED mean
  - gmean_y    : exp(mean(log(abs(x))))                  -> abs() guards negative values
  - wtd_gmean_y: exp(sum(p * log(abs(x))))
  - entropy_y  : normalizes on abs(x)/sum(abs(x)), natural log
  - wtd_entropy_y: normalizes on (p*abs(x))/sum(p*abs(x)), natural log
  - range_y    : max(x) - min(x)                          (unweighted)
  - wtd_range_y: max(p*x) - min(p*x)                       (elementwise product,
                                                             NOT (p_max*x_max)-(p_min*x_min))
  - p          : mole fraction = stoichiometric_count / sum(stoichiometric_counts)

Note: the recovered ElectronAffinity values already include Hamidieh's +1.5
offset baked in (recovered directly from the dataset), so no additional
offset is applied here. The recovered AtomicRadius values already reflect
his La/Ce manual overrides for the same reason.
"""

import re
import numpy as np
import pandas as pd


# ════════════════════════════════════════════════════════════════
# 1. ELEMENT PROPERTY TABLE
#    Order: [atomic_mass, atomic_radius(pm), valence, fie(kJ/mol),
#            electron_affinity(kJ/mol, +1.5 offset already applied by
#            Hamidieh), thermal_conductivity(W/mK), density(kg/m3),
#            fusion_heat(kJ/mol)]
#
#    77 elements: EXACTLY recovered from Final_dataset_83.csv by
#    least-squares back-solve (see module docstring). Every wtd_mean_X
#    column in Final_dataset_83.csv reconstructs with zero residual using
#    these values.
#
#    9 elements (He, Ne, Ar, Kr, Xe, Pm, Po, At, Rn): NEVER appear among
#    the 21,263 UCI compounds, so the back-solve is mathematically
#    undetermined for them (would otherwise be 0/0). These 9 use converted
#    fallback values instead, sourced from the standard CRC-style table
#    used earlier in this project (external_validation.ipynb) and
#    converted into this table's exact units/conventions:
#      - fie, ElectronAffinity: eV -> kJ/mol (x 96.485332)
#      - ElectronAffinity: +1.5 kJ/mol offset applied afterward, matching
#        the offset already baked into the 77 recovered values (verified
#        against known elements, e.g. recovered Cl EA = 350.5 kJ/mol =
#        real 349.0 kJ/mol + 1.5)
#      - Density: g/cm3 -> kg/m3 (x 1000)
#      - Valence: bond-count convention (e.g. noble gases = 0), not the
#        group-number convention a generic periodic table would give
#    These 9 elements essentially never occur in real superconductor
#    formulas (noble gases, or the highly unstable/radioactive Pm, Po, At,
#    Rn), so this fallback is a safety net for malformed input rather
#    than something expected to matter in practice -- but it now returns
#    physically reasonable numbers instead of zero if one appears.
#    CAVEAT: noble gases have genuine Valence = 0 (bond-count convention).
#    If one ever appears in an input formula, gmean_Valence/entropy_Valence
#    /wtd_gmean_Valence/wtd_entropy_Valence will raise a RuntimeWarning
#    (log(0)), since no real superconductor contains a noble gas this has
#    not been patched with an artificial non-zero Valence; flagged here so
#    it isn't mistaken for a bug if ever triggered by malformed input.
# ════════════════════════════════════════════════════════════════

ELEMENTAL_DATA = {
    'H': [1.007940, 53.000000, 1.000000, 1311.300000, 74.300000, 0.180500, 0.089900, 0.558000],
    'He': [4.003000, 31.000000, 0.000000, 2372.284900, 1.500000, 0.151300, 0.178500, 0.000000],
    'Li': [6.941000, 167.000000, 1.000000, 519.900000, 61.100000, 85.000000, 535.000000, 3.000000],
    'Be': [9.012182, 112.000000, 2.000000, 898.800000, 1.500000, 190.000000, 1848.000000, 7.950000],
    'B': [10.811000, 87.000000, 3.000000, 800.200000, 28.200000, 27.000000, 2460.000000, 50.000000],
    'C': [12.010700, 67.000000, 4.000000, 1085.700000, 155.400000, 140.000000, 2260.000000, 105.000000],
    'N': [14.006700, 56.000000, 3.000000, 1401.500000, 8.500000, 0.025830, 1.251000, 0.360000],
    'O': [15.999400, 48.000000, 2.000000, 1313.100000, 142.500000, 0.026580, 1.429000, 0.222000],
    'F': [18.998403, 42.000000, 1.000000, 1680.000000, 329.500000, 0.027700, 1.696000, 0.260000],
    'Ne': [20.180000, 38.000000, 0.000000, 2080.706200, 1.500000, 0.049300, 0.900200, 0.340000],
    'Na': [22.989769, 190.000000, 1.000000, 495.600000, 54.300000, 140.000000, 968.000000, 2.600000],
    'Mg': [24.305000, 145.000000, 2.000000, 737.300000, 1.500000, 160.000000, 1738.000000, 8.700000],
    'Al': [26.981539, 118.000000, 3.000000, 577.200000, 44.000000, 235.000000, 2700.000000, 10.700000],
    'Si': [28.085500, 111.000000, 4.000000, 786.000000, 135.100000, 150.000000, 2330.000000, 50.200000],
    'P': [30.973762, 98.000000, 5.000000, 1011.200000, 72.500000, 0.236000, 1823.000000, 0.640000],
    'S': [32.065000, 87.000000, 6.000000, 999.000000, 201.500000, 0.205000, 1960.000000, 1.730000],
    'Cl': [35.453000, 79.000000, 5.000000, 1254.900000, 350.500000, 0.008900, 3.214000, 3.200000],
    'Ar': [39.948000, 71.000000, 0.000000, 1520.608800, 1.500000, 0.017700, 1.783700, 1.188000],
    'K': [39.098300, 243.000000, 1.000000, 418.500000, 49.900000, 100.000000, 856.000000, 2.330000],
    'Ca': [40.078000, 194.000000, 2.000000, 589.400000, 3.870000, 200.000000, 1550.000000, 8.540000],
    'Sc': [44.955912, 184.000000, 3.000000, 630.800000, 19.600000, 16.000000, 2985.000000, 16.000000],
    'Ti': [47.867000, 176.000000, 4.000000, 657.800000, 9.100000, 22.000000, 4507.000000, 18.700000],
    'V': [50.941500, 171.000000, 5.000000, 650.100000, 52.100000, 31.000000, 6110.000000, 22.800000],
    'Cr': [51.996100, 166.000000, 6.000000, 652.400000, 65.800000, 94.000000, 7190.000000, 20.500000],
    'Mn': [54.938045, 161.000000, 4.000000, 716.800000, 1.500000, 7.700000, 7470.000000, 13.200000],
    'Fe': [55.845000, 156.000000, 3.000000, 759.100000, 17.200000, 79.000000, 7874.000000, 13.800000],
    'Co': [58.933195, 152.000000, 4.000000, 758.100000, 65.200000, 100.000000, 8900.000000, 16.200000],
    'Ni': [58.693400, 149.000000, 2.000000, 736.200000, 113.500000, 91.000000, 8908.000000, 17.200000],
    'Cu': [63.546000, 145.000000, 2.000000, 745.000000, 119.900000, 400.000000, 8960.000000, 13.100000],
    'Zn': [65.380000, 142.000000, 2.000000, 905.800000, 1.500000, 120.000000, 7140.000000, 7.350000],
    'Ga': [69.723000, 136.000000, 3.000000, 578.700000, 30.400000, 29.000000, 5904.000000, 5.590000],
    'Ge': [72.630000, 125.000000, 4.000000, 760.000000, 120.500000, 60.000000, 5323.000000, 31.800000],
    'As': [74.921600, 114.000000, 5.000000, 946.200000, 79.500000, 50.000000, 5727.000000, 27.700000],
    'Se': [78.960000, 103.000000, 6.000000, 940.400000, 196.500000, 0.520000, 4819.000000, 5.400000],
    'Br': [79.904000, 94.000000, 5.000000, 1142.000000, 326.100000, 0.120000, 3120.000000, 5.800000],
    'Kr': [83.798000, 188.000000, 0.000000, 1350.794600, 1.500000, 0.009400, 3.749300, 1.638000],
    'Rb': [85.467800, 265.000000, 1.000000, 402.800000, 48.400000, 58.000000, 1532.000000, 2.190000],
    'Sr': [87.620000, 219.000000, 2.000000, 549.000000, 6.530000, 35.000000, 2630.000000, 8.000000],
    'Y': [88.905850, 212.000000, 3.000000, 615.400000, 31.100000, 17.000000, 4472.000000, 11.400000],
    'Zr': [91.224000, 206.000000, 4.000000, 659.700000, 42.600000, 23.000000, 6511.000000, 21.000000],
    'Nb': [92.906380, 198.000000, 5.000000, 663.600000, 87.600000, 54.000000, 8570.000000, 26.800000],
    'Mo': [95.960000, 190.000000, 6.000000, 684.800000, 73.400000, 139.000000, 10280.000000, 36.000000],
    'Tc': [98.000000, 183.000000, 6.000000, 702.200000, 54.500000, 51.000000, 11500.000000, 23.000000],
    'Ru': [101.070000, 178.000000, 6.000000, 710.300000, 102.800000, 120.000000, 12370.000000, 25.700000],
    'Rh': [102.905500, 173.000000, 6.000000, 719.500000, 111.200000, 150.000000, 12450.000000, 21.700000],
    'Pd': [106.420000, 169.000000, 4.000000, 803.500000, 55.200000, 71.000000, 12023.000000, 16.700000],
    'Ag': [107.868200, 165.000000, 1.000000, 730.500000, 127.100000, 430.000000, 10490.000000, 11.300000],
    'Cd': [112.411000, 161.000000, 2.000000, 867.200000, 1.500000, 96.000000, 8650.000000, 6.300000],
    'In': [114.818000, 156.000000, 3.000000, 558.000000, 30.400000, 82.000000, 7310.000000, 3.260000],
    'Sn': [118.710000, 145.000000, 4.000000, 708.200000, 108.800000, 67.000000, 7310.000000, 7.000000],
    'Sb': [121.760000, 133.000000, 5.000000, 833.300000, 104.700000, 24.000000, 6697.000000, 19.700000],
    'Te': [127.600000, 123.000000, 6.000000, 869.000000, 191.700000, 3.000000, 6240.000000, 17.500000],
    'I': [126.904470, 115.000000, 7.000000, 1008.300000, 296.700000, 0.449000, 4940.000000, 7.760000],
    'Xe': [131.293000, 108.000000, 0.000000, 1170.367100, 1.500000, 0.005600, 5.890000, 2.300000],
    'Cs': [132.905452, 298.000000, 1.000000, 375.500000, 47.000000, 36.000000, 1879.000000, 2.090000],
    'Ba': [137.327000, 253.000000, 2.000000, 502.500000, 15.450000, 18.000000, 3510.000000, 8.000000],
    'La': [138.905470, 195.000000, 3.000000, 541.100000, 49.500000, 13.000000, 6146.000000, 6.300000],
    'Ce': [140.116000, 185.000000, 4.000000, 540.100000, 51.500000, 11.000000, 6689.000000, 5.500000],
    'Pr': [140.907650, 247.000000, 4.000000, 526.600000, 51.500000, 13.000000, 6640.000000, 6.900000],
    'Nd': [144.242000, 206.000000, 3.000000, 531.500000, 51.500000, 17.000000, 7010.000000, 7.100000],
    'Pm': [145.000000, 205.000000, 3.000000, 538.581100, 49.742700, 17.900000, 7260.000000, 7.130000],
    'Sm': [150.360000, 238.000000, 3.000000, 540.100000, 51.500000, 13.000000, 7353.000000, 8.500000],
    'Eu': [151.964000, 231.000000, 3.000000, 546.900000, 51.500000, 14.000000, 5244.000000, 9.300000],
    'Gd': [157.250000, 233.000000, 3.000000, 594.200000, 51.500000, 11.000000, 7901.000000, 10.000000],
    'Tb': [158.925350, 225.000000, 3.000000, 569.000000, 51.500000, 11.000000, 8219.000000, 10.800000],
    'Dy': [162.500000, 228.000000, 3.000000, 567.000000, 51.500000, 11.000000, 8551.000000, 11.100000],
    'Ho': [164.930320, 226.000000, 3.000000, 574.000000, 51.500000, 16.000000, 8795.000000, 17.000000],
    'Er': [167.259000, 226.000000, 3.000000, 581.000000, 51.500000, 15.000000, 9066.000000, 19.900000],
    'Tm': [168.934210, 222.000000, 3.000000, 589.000000, 51.500000, 17.000000, 9320.000000, 16.800000],
    'Yb': [173.054000, 222.000000, 3.000000, 603.000000, 51.500000, 39.000000, 6570.000000, 7.700000],
    'Lu': [174.966800, 217.000000, 3.000000, 513.000000, 51.500000, 16.000000, 9841.000000, 22.000000],
    'Hf': [178.490000, 208.000000, 4.000000, 575.200000, 1.500000, 23.000000, 13310.000000, 25.500000],
    'Ta': [180.947880, 200.000000, 5.000000, 760.100000, 32.500000, 57.000000, 16650.000000, 36.000000],
    'W': [183.840000, 193.000000, 6.000000, 769.700000, 80.100000, 170.000000, 19250.000000, 35.000000],
    'Re': [186.207000, 188.000000, 7.000000, 759.100000, 16.000000, 48.000000, 21020.000000, 33.000000],
    'Os': [190.230000, 185.000000, 6.000000, 819.800000, 107.600000, 87.000000, 22590.000000, 31.000000],
    'Ir': [192.217000, 180.000000, 6.000000, 868.100000, 152.500000, 150.000000, 22560.000000, 26.000000],
    'Pt': [195.084000, 177.000000, 6.000000, 868.100000, 206.800000, 71.000000, 21450.000000, 20.000000],
    'Au': [196.966569, 174.000000, 5.000000, 889.300000, 224.300000, 320.000000, 19300.000000, 12.500000],
    'Hg': [200.590000, 171.000000, 2.000000, 1006.000000, 1.500000, 8.300000, 13534.000000, 2.290000],
    'Tl': [204.383300, 156.000000, 3.000000, 588.900000, 20.700000, 46.000000, 11850.000000, 4.200000],
    'Pb': [207.200000, 154.000000, 4.000000, 715.200000, 36.600000, 35.000000, 11340.000000, 4.770000],
    'Bi': [208.980400, 143.000000, 5.000000, 702.900000, 92.700000, 8.000000, 9780.000000, 10.900000],
    'Po': [209.000000, 135.000000, 2.000000, 812.117000, 184.822100, 20.000000, 9320.000000, 13.000000],
    'At': [210.000000, 127.000000, 1.000000, 926.259200, 271.658900, 2.000000, 7000.000000, 6.000000],
    'Rn': [222.000000, 120.000000, 0.000000, 1036.734900, 1.500000, 0.003600, 9.730000, 3.000000],
}

# Column order matches ELEMENTAL_DATA lists.
# 'fie' kept as Hamidieh's short name (first ionization energy).
PROP_NAMES = [
    'atomic_mass', 'atomic_radius', 'Valence',
    'fie', 'ElectronAffinity', 'ThermalConductivity',
    'Density', 'FusionHeat',
]

# ════════════════════════════════════════════════════════════════
# 2. FORMULA PARSER
# ════════════════════════════════════════════════════════════════

def parse_formula(formula):
    """
    Parse a chemical formula string into {element: stoichiometric_count}.

    Elements with a zero coefficient (e.g. 'Bi0' in
    'Bi0Mo0.33Cu2.67Sr2Y1O7.41') are dropped, matching Hamidieh's own data
    cleaning: his R script comments explicitly flag such zero-coefficient
    entries as invalid and remove them. Keeping them would make that
    element's mole fraction exactly 0, which is a mathematical singularity
    for entropy (log(0)).
    """
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    matches = re.findall(pattern, str(formula))
    composition = {}
    for element, count in matches:
        if not element:
            continue
        if element not in ELEMENTAL_DATA:
            return None  # unknown / unsupported element
        count = float(count) if count else 1.0
        composition[element] = composition.get(element, 0.0) + count

    # Drop zero-coefficient elements (see docstring)
    composition = {e: c for e, c in composition.items() if c > 0}

    if not composition:
        return None
    return composition


# ════════════════════════════════════════════════════════════════
# 3. get_features() — EXACT port of the R function
# ════════════════════════════════════════════════════════════════

def get_features(y, p):
    """
    y : array of the raw elemental property values for the elements present
    p : array of mole fractions (same order as y), sums to 1

    Returns the 10 statistics in Hamidieh's fixed order:
    [mean, wtd_mean, gmean, wtd_gmean, entropy, wtd_entropy,
     range, wtd_range, std, wtd_std]
    """
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    mean_y = np.mean(y)
    wtd_mean_y = np.sum(p * y)

    gmean_y = np.exp(np.mean(np.log(np.abs(y))))
    wtd_gmean_y = np.exp(np.sum(p * np.log(np.abs(y))))

    tmp = np.abs(y) / np.sum(np.abs(y))
    entropy_y = -np.sum(tmp * np.log(tmp))

    tmp_w = (p * np.abs(y)) / np.sum(p * np.abs(y))
    wtd_entropy_y = -np.sum(tmp_w * np.log(tmp_w))

    range_y = np.max(y) - np.min(y)
    wtd_range_y = np.max(p * y) - np.min(p * y)   # elementwise product, R-exact

    std_y = np.sqrt(np.mean((y - mean_y) ** 2))            # population std (ddof=0)
    wtd_std_y = np.sqrt(np.sum((y - wtd_mean_y) ** 2 * p))  # centered on WEIGHTED mean

    return {
        'mean': mean_y, 'wtd_mean': wtd_mean_y,
        'gmean': gmean_y, 'wtd_gmean': wtd_gmean_y,
        'entropy': entropy_y, 'wtd_entropy': wtd_entropy_y,
        'range': range_y, 'wtd_range': wtd_range_y,
        'std': std_y, 'wtd_std': wtd_std_y,
    }


# ════════════════════════════════════════════════════════════════
# 4. TOP-30 SHAP-SELECTED COMPOSITIONAL FEATURES
#    (excludes upper_critical_field, coherence_length, vF_predicted_ms,
#     which are GL / vF-pipeline outputs, not raw compositional stats)
# ════════════════════════════════════════════════════════════════

# Each entry: (output_column_name, property_name, statistic_key)
TOP30_COMPOSITIONAL_SPEC = [
    ('range_ThermalConductivity',        'ThermalConductivity', 'range'),
    ('wtd_mean_Valence',                 'Valence',             'wtd_mean'),
    ('wtd_std_ThermalConductivity',      'ThermalConductivity', 'wtd_std'),
    ('wtd_gmean_Valence',                'Valence',             'wtd_gmean'),
    ('range_atomic_radius',              'atomic_radius',       'range'),
    ('std_atomic_mass',                  'atomic_mass',         'std'),
    ('wtd_entropy_ThermalConductivity',  'ThermalConductivity', 'wtd_entropy'),
    ('wtd_gmean_ThermalConductivity',    'ThermalConductivity', 'wtd_gmean'),
    ('wtd_std_ElectronAffinity',         'ElectronAffinity',    'wtd_std'),
    ('wtd_std_Valence',                  'Valence',             'wtd_std'),
    ('wtd_entropy_Density',              'Density',             'wtd_entropy'),
    ('std_ThermalConductivity',          'ThermalConductivity', 'std'),
    ('wtd_mean_Density',                 'Density',             'wtd_mean'),
    ('mean_Density',                     'Density',             'mean'),
    ('wtd_entropy_atomic_mass',          'atomic_mass',         'wtd_entropy'),
    ('wtd_std_fie',                      'fie',                 'wtd_std'),
    ('gmean_Density',                    'Density',             'gmean'),
    ('wtd_mean_ThermalConductivity',     'ThermalConductivity', 'wtd_mean'),
    ('wtd_entropy_FusionHeat',           'FusionHeat',          'wtd_entropy'),
    ('wtd_gmean_ElectronAffinity',       'ElectronAffinity',    'wtd_gmean'),
    ('wtd_mean_ElectronAffinity',        'ElectronAffinity',    'wtd_mean'),
    ('wtd_std_atomic_radius',            'atomic_radius',       'wtd_std'),
    ('wtd_entropy_atomic_radius',        'atomic_radius',       'wtd_entropy'),
    ('std_ElectronAffinity',             'ElectronAffinity',    'std'),
    ('wtd_std_atomic_mass',              'atomic_mass',         'wtd_std'),
    ('gmean_ElectronAffinity',           'ElectronAffinity',    'gmean'),
    ('wtd_entropy_fie',                  'fie',                 'wtd_entropy'),
]

# Sanity check: 27 compositional + 3 non-compositional (vF, xi, Hc2) = 30
assert len(TOP30_COMPOSITIONAL_SPEC) == 27, \
    f"Expected 27 compositional features, got {len(TOP30_COMPOSITIONAL_SPEC)}"

# Properties actually needed (skip computing stats for unused properties)
_NEEDED_PROPS = sorted(set(prop for _, prop, _ in TOP30_COMPOSITIONAL_SPEC))


def compute_composition_features(formula):
    """
    Compute the 27 SHAP-selected compositional statistics for one formula.
    Returns a dict {output_column_name: value}, or None if the formula
    could not be parsed (unknown element).
    """
    comp = parse_formula(formula)
    if comp is None:
        return None

    elements = list(comp.keys())
    counts = np.array([comp[e] for e in elements], dtype=float)
    p = counts / counts.sum()  # mole fractions

    # Compute all 10 stats once per needed property, then pick out the
    # specific (property, statistic) pairs the top-30 list actually uses.
    stats_by_prop = {}
    for prop in _NEEDED_PROPS:
        prop_idx = PROP_NAMES.index(prop)
        y = np.array([ELEMENTAL_DATA[e][prop_idx] for e in elements], dtype=float)
        stats_by_prop[prop] = get_features(y, p)

    out = {}
    for col_name, prop, stat_key in TOP30_COMPOSITIONAL_SPEC:
        out[col_name] = stats_by_prop[prop][stat_key]

    return out


# ════════════════════════════════════════════════════════════════
# 5. BATCH DRIVER
# ════════════════════════════════════════════════════════════════

def compute_features_for_dataframe(df, formula_col='material'):
    """
    df must have a column of chemical formula strings.
    Returns a new DataFrame with the 27 compositional feature columns,
    plus 'formula' and a 'parse_failed' flag for any row that couldn't
    be parsed (unknown element symbol).
    """
    rows = []
    failed_formulas = []

    for formula in df[formula_col]:
        feats = compute_composition_features(formula)
        if feats is None:
            failed_formulas.append(formula)
            rows.append({col: np.nan for col, _, _ in TOP30_COMPOSITIONAL_SPEC})
        else:
            rows.append(feats)

    out_df = pd.DataFrame(rows)
    out_df.insert(0, 'formula', df[formula_col].values)
    out_df['parse_failed'] = out_df['formula'].isin(failed_formulas)

    if failed_formulas:
        print(f"WARNING: {len(failed_formulas)} formulas failed to parse "
              f"(unknown element). Examples: {failed_formulas[:5]}")

    return out_df


def compute_composition_features_from_counts(element_counts):
    """
    Same as compute_composition_features(), but takes a dict/Series of
    {element_symbol: stoichiometric_count} directly instead of parsing a
    formula string. Use this when a pre-parsed stoichiometry table is
    available (e.g. unique_m.csv's per-element columns) -- it sidesteps
    regex parsing entirely and is the more robust path for a large batch.

    Zero-count elements should already be excluded (or will be filtered
    out here automatically).
    """
    elements = [e for e, c in element_counts.items()
                if e in ELEMENTAL_DATA and c and c > 0]
    if not elements:
        return None

    counts = np.array([element_counts[e] for e in elements], dtype=float)
    p = counts / counts.sum()

    stats_by_prop = {}
    for prop in _NEEDED_PROPS:
        prop_idx = PROP_NAMES.index(prop)
        y = np.array([ELEMENTAL_DATA[e][prop_idx] for e in elements], dtype=float)
        stats_by_prop[prop] = get_features(y, p)

    out = {}
    for col_name, prop, stat_key in TOP30_COMPOSITIONAL_SPEC:
        out[col_name] = stats_by_prop[prop][stat_key]
    return out


def compute_features_from_stoichiometry_table(stoich_df, element_cols=None):
    """
    Batch version using a wide stoichiometry table (one column per element,
    one row per compound), e.g. unique_m.csv from Hamidieh's R pipeline.

    stoich_df    : DataFrame, one row per compound
    element_cols : list of column names that are element symbols. If None,
                   auto-detects by intersecting stoich_df.columns with
                   ELEMENTAL_DATA keys.

    Returns a DataFrame of the 27 compositional features, row-aligned with
    stoich_df (same index / order), plus a 'parse_failed' flag.
    """
    if element_cols is None:
        element_cols = [c for c in stoich_df.columns if c in ELEMENTAL_DATA]

    rows = []
    failed_mask = []

    for _, row in stoich_df[element_cols].iterrows():
        counts = {e: row[e] for e in element_cols if row[e] > 0}
        feats = compute_composition_features_from_counts(counts)
        if feats is None:
            failed_mask.append(True)
            rows.append({col: np.nan for col, _, _ in TOP30_COMPOSITIONAL_SPEC})
        else:
            failed_mask.append(False)
            rows.append(feats)

    out_df = pd.DataFrame(rows, index=stoich_df.index)
    out_df['parse_failed'] = failed_mask

    n_failed = sum(failed_mask)
    if n_failed:
        print(f"WARNING: {n_failed} rows had no usable elements.")

    return out_df


def save_features_csv_from_formulas(
    output_path,
    input_csv,
    formula_col='name',
    label_col='Tc',
    label_out_col='critical_temp',
):
    """
    Compute the 27 SHAP-selected compositional features for an external
    dataset given as (formula, label) pairs -- e.g. the Stanev external
    validation set -- using the formula-string parser path.

    input_csv     : path to a CSV with a formula column and a label column
    formula_col   : name of the column holding chemical formula strings
                    (must use PROPER CASE element symbols, e.g. 'Ba0.4K0.6Fe2As2';
                    lowercase formulas will fail to parse)
    label_col     : name of the target column in input_csv (e.g. 'Tc')
    label_out_col : name to rename the label column to in the output
                    (default 'critical_temp', matching Final_dataset_83.csv)

    Rows with unparseable formulas (unknown element, or NaN formula/label)
    are dropped, with a warning printed showing the count and examples.
    """
    df_in = pd.read_csv(input_csv)
    n_before = len(df_in)

    df_in = df_in.dropna(subset=[formula_col, label_col]).reset_index(drop=True)
    n_after_na_drop = len(df_in)
    if n_before != n_after_na_drop:
        print(f"Dropped {n_before - n_after_na_drop} rows with missing "
              f"formula/label.")

    computed = compute_features_for_dataframe(df_in, formula_col=formula_col)

    # Attach label, drop failed-parse rows
    computed[label_out_col] = df_in[label_col].values
    n_failed = computed['parse_failed'].sum()
    if n_failed:
        failed_examples = computed.loc[computed['parse_failed'], 'formula'].head(5).tolist()
        print(f"WARNING: {n_failed} formulas failed to parse and were "
              f"dropped. Examples: {failed_examples}")
        computed = computed[~computed['parse_failed']].reset_index(drop=True)

    computed = computed.drop(columns=['parse_failed'])
    computed = computed.rename(columns={'formula': formula_col})

    # Reorder: formula, label, then the 27 features
    feature_cols = [c for c, _, _ in TOP30_COMPOSITIONAL_SPEC]
    computed = computed[[formula_col, label_out_col] + feature_cols]

    computed.to_csv(output_path, index=False)
    print(f"Saved {computed.shape[0]} rows x {computed.shape[1]} columns "
          f"-> {output_path}")

    return computed


def save_features_csv(
    output_path,
    stoich_path='unique_m - Copy.csv',
    labels_path='Final_dataset_83.csv',
):
    """
    Compute the 27 SHAP-selected compositional features for the full UCI
    dataset and save them to CSV, with 'material' and 'critical_temp'
    attached as identifier columns (row-order aligned with labels_path,
    since unique_m - Copy.csv and Final_dataset_83.csv share the same
    21,263-row order).

    output_path : where to write the resulting CSV
    stoich_path : path to unique_m.csv-style per-element stoichiometry table
    labels_path : path to the main dataset (for material name / critical_temp)
    """
    df_stoich = pd.read_csv(stoich_path)
    df_labels = pd.read_csv(labels_path)

    element_cols = [c for c in df_stoich.columns
                    if c not in ('material', 'critical_temp')]
    computed = compute_features_from_stoichiometry_table(
        df_stoich, element_cols=element_cols)

    computed.insert(0, 'material', df_labels['material'].values)
    computed.insert(1, 'critical_temp', df_labels['critical_temp'].values)

    n_failed = computed['parse_failed'].sum()
    computed = computed.drop(columns=['parse_failed'])

    computed.to_csv(output_path, index=False)
    print(f"Saved {computed.shape[0]} rows x {computed.shape[1]} columns "
          f"-> {output_path}")
    if n_failed:
        print(f"WARNING: {n_failed} rows had no usable elements.")

    return computed


if __name__ == '__main__':
    # ----------------------------------------------------------------
    # Formula-string self-test (sanity check on a few known compounds)
    # ----------------------------------------------------------------
    test_formulas = ['Re7Zr1', 'Y1Ba2Cu3O7', 'MgB2', 'Nb3Sn']
    test_df = pd.DataFrame({'material': test_formulas})
    result = compute_features_for_dataframe(test_df)
    pd.set_option('display.width', 200)
    pd.set_option('display.max_columns', 10)
    print("Formula-parser self-test:")
    print(result[['formula', 'mean_Density', 'wtd_mean_Valence',
                   'range_ThermalConductivity', 'parse_failed']])
    print()

    # ----------------------------------------------------------------
    # Full-dataset validation against Final_dataset_83.csv, if available.
    # Uses the stoichiometry-table path (unique_m.csv-style per-element
    # columns), which sidesteps regex parsing and is the recommended
    # route for the full 21,263-row UCI dataset.
    # ----------------------------------------------------------------
    import os
    stoich_path = 'unique_m - Copy.csv'
    actual_path = 'Final_dataset_83.csv'

    if os.path.exists(stoich_path) and os.path.exists(actual_path):
        from scipy.stats import pearsonr

        print("Validating against Final_dataset_83.csv ...")
        df_stoich = pd.read_csv(stoich_path)
        element_cols = [c for c in df_stoich.columns
                        if c not in ('material', 'critical_temp')]
        computed = compute_features_from_stoichiometry_table(
            df_stoich, element_cols=element_cols)

        actual = pd.read_csv(actual_path)
        cols_to_check = [c for c, _, _ in TOP30_COMPOSITIONAL_SPEC]

        print(f"\n{'Feature':<35}{'Pearson r':>12}{'Max abs err':>15}")
        print("-" * 62)
        for col in cols_to_check:
            r, _ = pearsonr(actual[col].values, computed[col].values)
            max_err = np.max(np.abs(actual[col].values - computed[col].values))
            print(f"{col:<35}{r:>12.6f}{max_err:>15.6f}")
    else:
        print("Final_dataset_83.csv / unique_m - Copy.csv not found; "
              "skipping full-dataset validation.")

In [ ]:
save_features_csv_from_formulas(
    output_path='stanev_250_uci_proportional.csv',
    input_csv='stanev_250_uci_proportional.csv',
    formula_col='material',   # e.g. 'material' or 'name'
    label_col='critical_temp',           # e.g. 'critical_temp' or 'Tc'
)

In [ ]:
# =============================================================================
#  External Validation — Stanev Dataset
#  Computes: family, subfamily, vF, coherence length (xi), upper critical field (Hc2)
#
#  Inputs  : stanev_recomputed.csv   ← your existing recomputed Stanev file
#                                       (must have 'material' and 'critical_temp'
#                                        columns plus the 81 Hamidieh features)
#            vF_hybrid_model.pkl     ← ML fallback model
#            vF_family_encoder.pkl   ← OneHotEncoder for family column
#
#  Outputs : stanev_with_GL_features.csv
# =============================================================================

import pandas as pd
import numpy as np
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA & MODELS
# =============================================================================
# ── adjust filename to match your actual Stanev CSV ──
df = pd.read_csv('stanev_250_uci_proportional.csv')

model   = joblib.load('vF_hybrid_model.pkl')
encoder = joblib.load('vF_family_encoder.pkl')

print(f"Stanev dataset shape : {df.shape}")
print(f"Columns sample       : {df.columns.tolist()[:5]} ...")
print(f"Tc range             : {df['critical_temp'].min():.2f} – "
      f"{df['critical_temp'].max():.2f} K\n")

# =============================================================================
# 2. CLASSIFICATION FUNCTIONS
#    Copied exactly from fermi_velocity_169_fixed_1.ipynb
# =============================================================================
def get_subfamily(mat_orig):
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    if has('Cu', mat) and has('O', mat):
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'

    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    if has('Cu', mat) and has('O', mat):
        if has('Ce', mat) and has('In', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']):
            return 'cuprate_LSCO'
        if has('Bi', mat) and has('Sr', mat):
            return 'cuprate_BSCCO'
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        if has('Ba', mat) and (has('Y', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'

    if has_major('Fe', mat):
        if has('As', mat) and has('O', mat):
            return 'iron_1111'
        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        if has('As', mat) and any(has(x, mat) for x in ['Ba', 'Sr', 'Ca', 'K', 'Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    if has('Ce', mat) and any(has(x, mat) for x in
                               ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and \
       any(has(r, mat) for r in RE2):
        return 'borocarbide'

    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) \
       and not has_major('Fe', mat):
        return 'conventional_A15'
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga']) \
       and not has_major('Fe', mat):
        return 'conventional_A15'

    if has('Nb', mat):
        return 'conventional_NbTi'

    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_NbTi':         'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# 3. vF LOOKUP TABLE (same canonical values as your training pipeline)
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'cuprate_other':             273400,
    'iron_1111':                 138000,
    'iron_122':                  121500,
    'iron_111':                  109000,
    'iron_11':                    95000,
    'iron_other':                121500,
    'MgB2_type':                 560000,
    'conventional_A15':          850000,
    'conventional_NbTi':         760000,
    'conventional_simple_metal': 1800000,
    'borocarbide':               165000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    'other':                     760000,
}

AMBIGUOUS_SUBFAMILIES = {'other', 'iron_other', 'cuprate_other'}

SINGLE_VALUE_FAMILIES = {
    'Chalcogenide': 95000,
    'Mg-B related': 560000,
}

VALID_VF = np.array(sorted(set(VF_LOOKUP.values())))


def snap_to_valid(pred_ms_array):
    pred_ms_array = np.asarray(pred_ms_array).reshape(-1, 1)
    diffs = np.abs(pred_ms_array - VALID_VF)
    return VALID_VF[diffs.argmin(axis=1)]


def apply_family_override(pred_ms_array, family_array):
    pred_ms_array = np.asarray(pred_ms_array).copy()
    family_array  = np.asarray(family_array)
    for fam, val in SINGLE_VALUE_FAMILIES.items():
        pred_ms_array[family_array == fam] = val
    return pred_ms_array


# =============================================================================
# 4. ASSIGN FAMILY & SUBFAMILY
# =============================================================================
# material column — adjust name if yours is different (e.g. 'formula', 'name')
MATERIAL_COL = 'material'

df['subfamily'] = df[MATERIAL_COL].apply(get_subfamily)
df['family']    = df[MATERIAL_COL].apply(get_family)
df['is_ambiguous'] = df['subfamily'].isin(AMBIGUOUS_SUBFAMILIES)

print("Family distribution (Stanev):")
print(df['family'].value_counts().to_string())
print(f"\nSubfamily distribution:")
print(df['subfamily'].value_counts().to_string())
print(f"\nAmbiguous rows : {df['is_ambiguous'].sum()} "
      f"({df['is_ambiguous'].mean()*100:.1f}%)")

# =============================================================================
# 5. PREDICT vF — hybrid: direct lookup + ML fallback
# =============================================================================
NUMERIC_FEATURES = [
    "wtd_std_fie",
    "wtd_entropy_ElectronAffinity",
    "wtd_std_ThermalConductivity",
    "wtd_entropy_fie",
    "wtd_std_FusionHeat",
    "range_Valence",
    "mean_ElectronAffinity",
    "wtd_std_Valence",
    "wtd_mean_FusionHeat",
    "wtd_entropy_FusionHeat",
    "wtd_gmean_Density",
    "wtd_mean_Valence",
]

# check all required numeric features are present
missing = [f for f in NUMERIC_FEATURES if f not in df.columns]
if missing:
    print(f"\nWARNING: Missing features for ML fallback: {missing}")
    print("These must be present in your recomputed Stanev CSV.")
    print("ML fallback rows will default to subfamily lookup value.\n")

vf_pred = np.zeros(len(df))
pred_source = np.array(['lookup'] * len(df), dtype=object)

# --- non-ambiguous: direct lookup ---
lookup_mask = ~df['is_ambiguous'].values
vf_pred[lookup_mask] = df.loc[lookup_mask, 'subfamily'].map(VF_LOOKUP).values

# --- ambiguous: ML fallback ---
amb_mask = df['is_ambiguous'].values
if amb_mask.sum() > 0 and len(missing) == 0:
    fam_enc  = encoder.transform(df.loc[amb_mask, ['family']])
    X_num    = df.loc[amb_mask, NUMERIC_FEATURES].values
    X_amb    = np.hstack([X_num, fam_enc])
    raw_log  = model.predict(X_amb)
    raw_ms   = 10 ** raw_log
    snapped  = snap_to_valid(raw_ms)
    snapped  = apply_family_override(snapped, df.loc[amb_mask, 'family'].values)
    vf_pred[amb_mask]     = snapped
    pred_source[amb_mask] = 'model'
elif amb_mask.sum() > 0 and len(missing) > 0:
    # fallback to 'other' lookup value when features missing
    vf_pred[amb_mask] = VF_LOOKUP['other']
    pred_source[amb_mask] = 'lookup_default'

df['vF_predicted_ms']   = vf_pred.astype(int)
df['prediction_source'] = pred_source

print(f"\nvF prediction source breakdown:")
print(df['prediction_source'].value_counts().to_string())
print(f"\nvF range: {df['vF_predicted_ms'].min():,} – "
      f"{df['vF_predicted_ms'].max():,} m/s")

# =============================================================================
# 6. COMPUTE GL FEATURES
# =============================================================================
Kb    = 1.380649e-23      # J/K
Pi    = np.pi
Phi0  = 2.067833848e-15   # Wb
h_cut = 1.054571817e-34   # J·s

Tc = np.clip(df['critical_temp'].values, 0.01, None)
vF = df['vF_predicted_ms'].values

energy_gap = 1.764 * Kb * Tc
df['coherence_length']     = (h_cut * vF) / (Pi * energy_gap)   # metres
df['upper_critical_field'] = Phi0 / (2 * Pi * df['coherence_length'] ** 2)  # Tesla

print(f"\nGL features computed:")
print(f"  xi  range : {df['coherence_length'].min():.3e} – "
      f"{df['coherence_length'].max():.3e} m")
print(f"  Hc2 range : {df['upper_critical_field'].min():.3f} – "
      f"{df['upper_critical_field'].max():.3f} T")

# =============================================================================
# 7. SANITY CHECK — family-level vF summary
# =============================================================================
print("\nPer-family vF summary:")
summary = df.groupby('family')['vF_predicted_ms'].agg(
    n='count', mean='mean', std='std', min='min', max='max'
).round(0)
print(summary.to_string())


In [ ]:


# =============================================================================
# 8. SAVE OUTPUT
# =============================================================================
df.to_csv('stanev_250_new.csv', index=False)
print("\nSaved -> stanev_250_new.csv")
print(f"Final shape: {df.shape}")
print("\nDone.")

In [ ]:
import pandas as pd
import numpy as np

# Load 5377 unseen compounds file
df_250 = pd.read_csv('stanev_250_new.csv') 

print("250 COMPOUND critical_temp DISTRIBUTION:")
bins = [(0,10),(10,30),(30,50),(50,80),(80,100),(100,130), (130,160), (160,200), (200,250), (250,300)]
for lo,hi in bins:
    n = ((df_250['critical_temp']>=lo)&(df_250['critical_temp']<hi)).sum()
    pct = 100*n/len(df_250)
    print(f"  {lo}-{hi}K: {n} ({pct:.1f}%)")
print(f"\nMean critical_temp: {df_250['critical_temp'].mean():.2f} K")
print(f"Max critical_temp:  {df_250['critical_temp'].max():.2f} K")
print(f"Std critical_temp:  {df_250['critical_temp'].std():.2f} K")

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import joblib

# 1. Load my saved model and scaler
my_model = joblib.load("stacking_model_30features.pkl")
scaler     = joblib.load("robust_scaler.pkl")

# 2. Get my top 30 feature names again (just to be safe)
df_top30 = pd.read_csv("top30_vF_xi_Hc_train_83.csv")
top30_cols = [c for c in df_top30.columns if c != "critical_temp"]

# 3. Handle NaNs the EXACT same way we did in training (fillna with 0)
# We fill NaNs before converting to .values so Pandas handles it cleanly
df_external_clean = df_250.copy()
df_external_clean[top30_cols] = df_external_clean[top30_cols].fillna(0)

# 4. Prepare features and target
X_ext = df_external_clean[top30_cols].values
y_ext = df_external_clean["critical_temp"].values

# 5. Scale using my scaler (using .transform() only!)
X_ext_scaled = scaler.transform(X_ext)

# 6. Predict!
y_pred = my_model.predict(X_ext_scaled)

# print the final metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

r2 = r2_score(y_ext, y_pred)
rmse = np.sqrt(mean_squared_error(y_ext, y_pred))
mae = mean_absolute_error(y_ext, y_pred)

print(f"--- FINAL TRUE EXTERNAL VALIDATION ---")
print(f"Compounds tested: {len(y_ext)}")
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.4f} K")
print(f"MAE:  {mae:.4f} K")
